# Stochastic Gradient Descent — a mathematical exploration

*I've spent time around people who use SGD every day. I understand what it does at a high level — it finds the minimum of a loss function during training. What I want to understand is the machinery underneath: why the update rule is what it is, what the 'stochastic' part actually buys you beyond computational convenience, and what the pathologies look like when you can see them geometrically.*

*I'm going to keep this grounded in linear regression throughout. Not because it's the most exciting application, but because the geometry is fully visible in 2D and 3D, which means I can see exactly what the optimiser is doing on every step.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import FancyArrowPatch
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# Consistent style throughout
plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor': '#16213e',
    'axes.edgecolor': '#444',
    'axes.labelcolor': '#ccc',
    'xtick.color': '#888',
    'ytick.color': '#888',
    'text.color': '#ddd',
    'grid.color': '#333',
    'grid.alpha': 0.5,
    'font.family': 'monospace',
    'axes.titlecolor': '#e05c5c',
    'axes.titlesize': 13,
    'axes.titlepad': 10,
})

ACCENT = '#e05c5c'
BLUE   = '#4fc3f7'
GREEN  = '#81c784'
YELLOW = '#fff176'

rng = np.random.default_rng(42)
print('Ready.')

---

## 1. The problem: what are we actually minimising?

Start with a dataset. I want a linear relationship with some noise, so I have something real to fit against.

The model is $\hat{y} = w \cdot x + b$ — just a line, parameterised by slope $w$ and intercept $b$.

The **loss** I'll use is Mean Squared Error:

$$\mathcal{L}(w, b) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = \frac{1}{n} \sum_{i=1}^{n} (y_i - wx_i - b)^2$$

This is a scalar function of two parameters. That means it defines a **surface** in $(w, b, \mathcal{L})$ space — a loss landscape. The task of training is to find the lowest point on that surface.

In [ ]:
# Generate data: y = 2x + 1 + noise
TRUE_W, TRUE_B = 2.0, 1.0
n = 80
X = rng.uniform(-3, 3, n)
y = TRUE_W * X + TRUE_B + rng.normal(0, 1.2, n)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('The dataset and its loss landscape', color=ACCENT, fontsize=14)

# Left: scatter
ax = axes[0]
ax.scatter(X, y, color=BLUE, alpha=0.6, s=20, label='data')
xline = np.linspace(-3, 3, 100)
ax.plot(xline, TRUE_W * xline + TRUE_B, color=ACCENT, lw=2, label=f'true line: y={TRUE_W}x+{TRUE_B}')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('The data')
ax.legend(fontsize=9)
ax.grid(True)

# Right: loss surface as a contour plot (top-down view of the bowl)
ax = axes[1]
w_grid = np.linspace(-1, 5, 200)
b_grid = np.linspace(-3, 5, 200)
W, B = np.meshgrid(w_grid, b_grid)

# Vectorised MSE over the parameter grid
# X is (n,), W is (200,200) — broadcast via newaxis
preds = W[:, :, np.newaxis] * X[np.newaxis, np.newaxis, :] + B[:, :, np.newaxis]
L = np.mean((y[np.newaxis, np.newaxis, :] - preds) ** 2, axis=2)

cp = ax.contourf(W, B, L, levels=40, cmap='inferno')
ax.contour(W, B, L, levels=15, colors='white', alpha=0.2, linewidths=0.5)
plt.colorbar(cp, ax=ax, label='MSE loss')

# Mark the true minimum
ax.plot(TRUE_W, TRUE_B, '*', color=GREEN, markersize=14, label=f'true minimum (w={TRUE_W}, b={TRUE_B})', zorder=5)
ax.set_xlabel('w (slope)'); ax.set_ylabel('b (intercept)')
ax.set_title('Loss landscape (top-down view)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'The loss landscape is a bowl — convex. There is exactly one minimum.')
print(f'For linear regression with MSE this is always the case.')
print(f'For neural networks it is emphatically not.')

That bright yellow star is the lowest point of the bowl — the parameter values that best explain the data. The contour lines are like elevation lines on a map: wherever you stand on this surface, the gradient points uphill most steeply, and its negative points you toward the bottom.

---

## 2. Gradient descent: the update rule and why it works

The gradient of $\mathcal{L}$ with respect to the parameters tells us the direction of steepest ascent. We want descent, so we move in the *opposite* direction:

$$\theta_{t+1} = \theta_t - \eta \cdot \nabla_\theta \mathcal{L}(\theta_t)$$

where $\theta = (w, b)$ and $\eta$ is the **learning rate** — a scalar that controls step size.

For our MSE loss, the partial derivatives are:

$$\frac{\partial \mathcal{L}}{\partial w} = -\frac{2}{n} \sum_{i=1}^{n} x_i (y_i - wx_i - b)$$

$$\frac{\partial \mathcal{L}}{\partial b} = -\frac{2}{n} \sum_{i=1}^{n} (y_i - wx_i - b)$$

Nothing exotic — just the chain rule applied to a sum of squares. The residual $(y_i - \hat{y}_i)$ is the error on example $i$, and the gradient weights each error by the corresponding input $x_i$ (for $w$) or by 1 (for $b$).

In [ ]:
def mse_loss(w, b, X, y):
    return np.mean((y - w * X - b) ** 2)

def mse_gradients(w, b, X, y):
    """Returns (dL/dw, dL/db) for the full dataset."""
    residuals = y - w * X - b
    dw = -2 * np.mean(X * residuals)
    db = -2 * np.mean(residuals)
    return dw, db

def gradient_descent(X, y, w_init, b_init, lr, n_steps):
    """Full-batch gradient descent. Uses ALL data points every step."""
    w, b = w_init, b_init
    history = [(w, b, mse_loss(w, b, X, y))]
    for _ in range(n_steps):
        dw, db = mse_gradients(w, b, X, y)
        w -= lr * dw
        b -= lr * db
        history.append((w, b, mse_loss(w, b, X, y)))
    return np.array(history)

# Run from a bad starting point
w0, b0 = -0.5, 4.0
history_gd = gradient_descent(X, y, w0, b0, lr=0.05, n_steps=80)

print(f'Start:  w={w0:.2f}, b={b0:.2f}, loss={history_gd[0, 2]:.3f}')
print(f'End:    w={history_gd[-1,0]:.3f}, b={history_gd[-1,1]:.3f}, loss={history_gd[-1,2]:.4f}')
print(f'Target: w={TRUE_W:.3f}, b={TRUE_B:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Full-batch gradient descent', color=ACCENT, fontsize=14)

# Left: path through the loss landscape
ax = axes[0]
ax.contourf(W, B, L, levels=40, cmap='inferno', alpha=0.85)
ax.contour(W, B, L, levels=15, colors='white', alpha=0.2, linewidths=0.5)
ws, bs = history_gd[:, 0], history_gd[:, 1]
ax.plot(ws, bs, '-o', color=BLUE, lw=1.5, markersize=3, alpha=0.9, label='GD path')
ax.plot(ws[0], bs[0], 's', color=YELLOW, markersize=10, label='start', zorder=5)
ax.plot(ws[-1], bs[-1], '*', color=GREEN, markersize=14, label='end', zorder=5)
ax.plot(TRUE_W, TRUE_B, '+', color='white', markersize=12, lw=2, label='true min', zorder=5)
ax.set_xlabel('w'); ax.set_ylabel('b')
ax.set_title('Path through parameter space')
ax.legend(fontsize=9)

# Right: loss over iterations
ax = axes[1]
ax.plot(history_gd[:, 2], color=ACCENT, lw=2)
ax.set_xlabel('iteration')
ax.set_ylabel('MSE loss')
ax.set_title('Loss over time')
ax.grid(True)

plt.tight_layout()
plt.show()

Clean. The path follows the bowl's curvature down to the minimum, and the loss curve is smooth and monotonically decreasing.

This is **full-batch gradient descent** (sometimes just called GD, or batch GD). Every step uses the gradient computed over *all* $n$ examples. That gradient is exact — it's the true gradient of the loss surface at that point.

### The learning rate matters — a lot

Let's see what happens when we pick a learning rate that's too large.

In [ ]:
# Three different learning rates — same starting point
lrs = [0.005, 0.05, 0.3]
labels = ['lr=0.005 (too slow)', 'lr=0.05 (good)', 'lr=0.3 (too fast → diverges)']
colors = [BLUE, GREEN, ACCENT]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Learning rate sensitivity', color=ACCENT, fontsize=14)

ax_landscape = axes[0]
ax_landscape.contourf(W, B, L, levels=40, cmap='inferno', alpha=0.7)
ax_landscape.contour(W, B, L, levels=15, colors='white', alpha=0.2, linewidths=0.5)

ax_loss = axes[1]

for lr, label, color in zip(lrs, labels, colors):
    h = gradient_descent(X, y, w0, b0, lr=lr, n_steps=80)
    ws, bs, losses = h[:, 0], h[:, 1], h[:, 2]
    
    ax_landscape.plot(ws, bs, '-', color=color, lw=1.5, alpha=0.85, label=label)
    ax_landscape.plot(ws[0], bs[0], 's', color=YELLOW, markersize=8, zorder=5)
    
    # Clip loss for visibility when it explodes
    clipped = np.clip(losses, 0, 30)
    ax_loss.plot(clipped, color=color, lw=1.8, label=label)

ax_landscape.plot(TRUE_W, TRUE_B, '+', color='white', markersize=12, lw=2, zorder=5)
ax_landscape.set_xlabel('w'); ax_landscape.set_ylabel('b')
ax_landscape.set_title('Paths (all same start)')
ax_landscape.legend(fontsize=8)

ax_loss.set_xlabel('iteration')
ax_loss.set_ylabel('MSE loss (clipped at 30)')
ax_loss.set_title('Loss curves')
ax_loss.legend(fontsize=8)
ax_loss.grid(True)

plt.tight_layout()
plt.show()

---

## 3. The stochastic part — and why noise is not just a compromise

**Full-batch GD** computes $\nabla \mathcal{L}$ using all $n$ examples. For $n = 80$ that's fine. For $n = 80{,}000{,}000$ it's a single gradient step that costs more computation than most of us have time for.

**Stochastic GD (SGD)** instead picks a *single random example* at each step and computes the gradient using only that example:

$$\theta_{t+1} = \theta_t - \eta \cdot \nabla_\theta \mathcal{L}(\theta_t; x_i, y_i)$$

This gradient is a **noisy estimate** of the true gradient. It's the right direction *on average*, but for any individual step it will be wrong by some amount — the error depending on how unrepresentative that single example is.

Why does this work? Two reasons:

1. **Law of large numbers.** Over many steps, the noise averages out. If you take 10,000 noisy steps, the accumulated movement is close to where 10,000 clean steps would have taken you.

2. **The noise is actually useful in non-convex settings.** This is the subtle part. In a bumpy landscape (which neural networks have), a gradient that's too clean will follow every valley precisely — including ones that dead-end in local minima. Noisy gradients have enough variance to occasionally kick the optimiser out of shallow traps. This is a genuine benefit, not just an acceptable cost.

For our convex bowl, we won't see benefit 2. But we *will* see the characteristic SGD behaviour: a jagged, noisy path that still converges.

In [ ]:
def sgd(X, y, w_init, b_init, lr, n_steps, batch_size=1):
    """
    SGD (batch_size=1) or mini-batch SGD (batch_size > 1).
    Shuffles data each pass through, then samples without replacement.
    """
    w, b = w_init, b_init
    history = [(w, b, mse_loss(w, b, X, y))]
    n = len(X)
    indices = np.arange(n)
    
    for step in range(n_steps):
        # Re-shuffle at the start of each epoch
        if step % n == 0:
            rng.shuffle(indices)
        
        # Sample a mini-batch
        batch_idx = indices[(step % n): (step % n) + batch_size]
        X_batch = X[batch_idx]
        y_batch = y[batch_idx]
        
        # Gradient on this batch only
        dw, db = mse_gradients(w, b, X_batch, y_batch)
        w -= lr * dw
        b -= lr * db
        
        # Store full-dataset loss (so we can compare fairly)
        history.append((w, b, mse_loss(w, b, X, y)))
    
    return np.array(history)

# Same start, same learning rate, different algorithms
history_sgd = sgd(X, y, w0, b0, lr=0.05, n_steps=400, batch_size=1)

print('SGD (batch=1):')
print(f'  Final w={history_sgd[-1,0]:.3f}, b={history_sgd[-1,1]:.3f}, loss={history_sgd[-1,2]:.4f}')
print(f'  GD needed 80 steps. SGD used 400 — but each step was {n}x cheaper to compute.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('GD vs SGD — same landscape, different paths', color=ACCENT, fontsize=14)

ax = axes[0]
ax.contourf(W, B, L, levels=40, cmap='inferno', alpha=0.7)
ax.contour(W, B, L, levels=15, colors='white', alpha=0.15, linewidths=0.5)

# GD path
ax.plot(history_gd[:, 0], history_gd[:, 1],
        '-o', color=GREEN, lw=1.5, markersize=2, alpha=0.9, label='GD (80 steps)')

# SGD path — show first 200 steps so it's legible
ax.plot(history_sgd[:200, 0], history_sgd[:200, 1],
        '-', color=ACCENT, lw=0.8, alpha=0.7, label='SGD (first 200 steps)')

ax.plot(w0, b0, 's', color=YELLOW, markersize=10, label='start', zorder=5)
ax.plot(TRUE_W, TRUE_B, '+', color='white', markersize=14, lw=2.5, label='true min', zorder=5)
ax.set_xlabel('w'); ax.set_ylabel('b')
ax.set_title('Paths through parameter space')
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(history_gd[:, 2], color=GREEN, lw=2, label='GD')
# For SGD, smooth the loss curve so the comparison is fair — the jaggedness
# is real but obscures the trend
window = 20
sgd_smoothed = np.convolve(history_sgd[:, 2],
                            np.ones(window)/window, mode='valid')
ax.plot(history_sgd[:, 2], color=ACCENT, lw=0.6, alpha=0.3, label='SGD (raw)')
ax.plot(np.arange(window-1, len(history_sgd)), sgd_smoothed,
        color=ACCENT, lw=2, label=f'SGD (smoothed, window={window})')

ax.set_xlabel('step')
ax.set_ylabel('MSE loss (full dataset)')
ax.set_title('Loss — note SGD is noisy but trends the same way')
ax.legend(fontsize=8)
ax.grid(True)

plt.tight_layout()
plt.show()

The SGD path looks like someone walking toward the minimum in a fog — they're heading roughly the right direction, but each individual step is deflected by whatever single data point they happened to look at. The *trend* is correct; the individual steps are not.

Notice also: the loss curve for SGD is jagged in a way GD's isn't. This is because after each noisy step we evaluate the loss on the *full* dataset — so we're measuring a globally-correct thing but after a locally-noisy update. The loss bounces around the minimum rather than converging cleanly to it.

---

## 4. Mini-batch SGD — the thing everyone actually uses

Pure SGD (batch size = 1) is noisy. Full-batch GD is slow. Mini-batch SGD picks a middle ground: at each step, sample a batch of $k$ examples ($k$ is typically 32, 64, or 256 in practice), compute the gradient over that batch, and update.

The gradient over a batch of $k$ examples:

$$\nabla_\theta \mathcal{L}_\text{batch} = \frac{1}{k} \sum_{i \in \mathcal{B}} \nabla_\theta \ell(\theta; x_i, y_i)$$

As $k \to n$ this approaches the true gradient (full-batch GD). As $k \to 1$ this becomes pure SGD. The batch size is a **variance-computation tradeoff**.

In [ ]:
batch_sizes = [1, 4, 16, 80]  # 80 = full batch
batch_labels = ['batch=1 (pure SGD)', 'batch=4', 'batch=16', 'batch=80 (full GD)']
batch_colors = [ACCENT, '#ff9966', YELLOW, GREEN]
n_steps = 500

histories = {}
for bs in batch_sizes:
    histories[bs] = sgd(X, y, w0, b0, lr=0.05, n_steps=n_steps, batch_size=bs)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Effect of batch size on convergence', color=ACCENT, fontsize=14)

ax = axes[0]
ax.contourf(W, B, L, levels=40, cmap='inferno', alpha=0.7)
ax.contour(W, B, L, levels=15, colors='white', alpha=0.15, linewidths=0.5)
for bs, label, color in zip(batch_sizes, batch_labels, batch_colors):
    h = histories[bs]
    ax.plot(h[:150, 0], h[:150, 1], '-', color=color, lw=1, alpha=0.8, label=label)
ax.plot(w0, b0, 's', color=YELLOW, markersize=10, zorder=5)
ax.plot(TRUE_W, TRUE_B, '+', color='white', markersize=14, lw=2.5, zorder=5)
ax.set_xlabel('w'); ax.set_ylabel('b')
ax.set_title('Paths (first 150 steps)')
ax.legend(fontsize=8)

ax = axes[1]
for bs, label, color in zip(batch_sizes, batch_labels, batch_colors):
    losses = histories[bs][:, 2]
    if bs < 16:
        window = 30
        smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
        ax.plot(losses, color=color, lw=0.4, alpha=0.25)
        ax.plot(np.arange(window-1, len(losses)), smoothed, color=color, lw=2, label=label)
    else:
        ax.plot(losses, color=color, lw=2, label=label)

ax.set_xlabel('step')
ax.set_ylabel('MSE loss')
ax.set_title('Loss curves (smoothed for noisy series)')
ax.legend(fontsize=8)
ax.grid(True)

plt.tight_layout()
plt.show()

---

## 5. Pathologies: what can go wrong

The convex bowl is a best-case scenario. Let me engineer some harder surfaces and watch what happens.

### 5a. Saddle points

A **saddle point** is a critical point (gradient = 0) that is neither a minimum nor a maximum — it curves upward in some directions and downward in others. In high-dimensional spaces (real neural networks) saddle points are *far more common* than local minima. The gradient is zero there, so gradient descent stalls.

In [ ]:
# Construct a surface with a saddle point at origin: f(x,y) = x^2 - y^2
# gradient: (2x, -2y). At (0,0): gradient = 0. But it's a saddle, not a min.

def saddle_loss(w, b):
    return w**2 - b**2

def saddle_grad(w, b):
    return 2*w, -2*b

ws = np.linspace(-2, 2, 200)
bs = np.linspace(-2, 2, 200)
WS, BS = np.meshgrid(ws, bs)
LS = saddle_loss(WS, BS)

# Two trajectories: one approaching along the valley (will stall),
# one with noise (may escape)
def run_on_saddle(start_w, start_b, lr, n_steps, noise_scale=0.0):
    w, b = start_w, start_b
    path = [(w, b)]
    for _ in range(n_steps):
        dw, db = saddle_grad(w, b)
        # Optionally inject noise to simulate SGD
        dw += rng.normal(0, noise_scale)
        db += rng.normal(0, noise_scale)
        w -= lr * dw
        b -= lr * db
        path.append((w, b))
    return np.array(path)

path_clean = run_on_saddle(0.01, 1.5, lr=0.1, n_steps=60, noise_scale=0.0)
path_noisy = run_on_saddle(0.01, 1.5, lr=0.1, n_steps=60, noise_scale=0.3)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Saddle points: clean GD stalls, noisy SGD can escape', color=ACCENT, fontsize=14)

for ax, path, title in zip(axes,
                             [path_clean, path_noisy],
                             ['Clean GD (no noise)', 'SGD (gradient noise)']):
    cf = ax.contourf(WS, BS, LS, levels=40, cmap='RdBu_r', alpha=0.8)
    ax.contour(WS, BS, LS, levels=15, colors='white', alpha=0.2, linewidths=0.4)
    ax.plot(path[:, 0], path[:, 1], '-o', color=YELLOW, lw=1.5, markersize=3, label='path')
    ax.plot(path[0, 0], path[0, 1], 's', color=GREEN, markersize=10, label='start', zorder=5)
    ax.plot(path[-1, 0], path[-1, 1], 'v', color=ACCENT, markersize=10, label='end', zorder=5)
    ax.plot(0, 0, '+', color='white', markersize=14, lw=2, label='saddle point', zorder=5)
    ax.set_xlabel('w'); ax.set_ylabel('b')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.set_xlim(-2.1, 2.1); ax.set_ylim(-2.1, 2.1)

plt.tight_layout()
plt.show()

print(f'Clean GD final position:  w={path_clean[-1,0]:.4f}, b={path_clean[-1,1]:.4f}')
print(f'Noisy SGD final position: w={path_noisy[-1,0]:.4f}, b={path_noisy[-1,1]:.4f}')
print()
print('The saddle surface is flat in the w-direction at w~0, so a trajectory')
print('approaching from near w=0 gets pulled toward the saddle and stalls.')
print('SGD gradient noise perturbs w enough to escape downhill.')

### 5b. Ill-conditioned landscapes (elongated bowls)

The MSE bowl above was fairly round. In real problems, the loss landscape is often highly **elongated** — one direction changes the loss much faster than another. This is called **ill-conditioning**, and it causes gradient descent to oscillate: the gradient points across the steep axis rather than down the long valley.

In [ ]:
# Generate an ill-conditioned dataset: highly correlated features
# This creates an elongated loss bowl
rng2 = np.random.default_rng(7)
n_ill = 100
X_ill = rng2.uniform(0, 10, n_ill)
y_ill = 0.5 * X_ill + rng2.normal(0, 0.1, n_ill)  # Very low noise: tight correlation

# Compute loss landscape for this dataset
w_ill = np.linspace(-1, 2, 200)
b_ill = np.linspace(-2, 2, 200)
W_ill, B_ill = np.meshgrid(w_ill, b_ill)
preds_ill = W_ill[:, :, np.newaxis] * X_ill[np.newaxis, np.newaxis, :] + B_ill[:, :, np.newaxis]
L_ill = np.mean((y_ill[np.newaxis, np.newaxis, :] - preds_ill) ** 2, axis=2)

# Gradient descent with a moderately large lr
def gd_generic(loss_fn, grad_fn, w_init, b_init, lr, n_steps):
    w, b = w_init, b_init
    path = [(w, b, loss_fn(w, b))]
    for _ in range(n_steps):
        dw, db = grad_fn(w, b)
        w -= lr * dw
        b -= lr * db
        path.append((w, b, loss_fn(w, b)))
    return np.array(path)

loss_fn = lambda w, b: mse_loss(w, b, X_ill, y_ill)
grad_fn = lambda w, b: mse_gradients(w, b, X_ill, y_ill)

path_slow = gd_generic(loss_fn, grad_fn, 1.5, 1.5, lr=0.003, n_steps=200)
path_osc  = gd_generic(loss_fn, grad_fn, 1.5, 1.5, lr=0.015, n_steps=80)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Ill-conditioned landscape — oscillation vs slow convergence', color=ACCENT, fontsize=14)

for ax, path, label, color in zip(axes,
                                    [path_slow, path_osc],
                                    ['lr=0.003 (slow but stable)', 'lr=0.015 (oscillates across steep axis)'],
                                    [GREEN, ACCENT]):
    ax.contourf(W_ill, B_ill, L_ill, levels=40, cmap='inferno', alpha=0.8)
    ax.contour(W_ill, B_ill, L_ill, levels=20, colors='white', alpha=0.15, linewidths=0.4)
    ax.plot(path[:, 0], path[:, 1], '-', color=color, lw=1.2, alpha=0.9, label=label)
    ax.plot(path[0, 0], path[0, 1], 's', color=YELLOW, markersize=10, zorder=5)
    ax.plot(path[-1, 0], path[-1, 1], 'v', color='white', markersize=8, zorder=5)
    ax.set_xlabel('w'); ax.set_ylabel('b')
    ax.set_title(label)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print('Ill-conditioning is why momentum, Adam, and RMSProp were invented.')
print('They adapt the effective learning rate per parameter based on gradient history.')
print('But that is a separate topic.')

---

## 6. Putting it together: the geometry of what SGD is doing

Let me finish with a single clean visualisation that captures the whole story: the loss surface, a trajectory, and the gradient vectors that cause each step.

In [ ]:
# Show the gradient vector field alongside the SGD path
# Sample gradient vectors on a coarse grid
w_gv = np.linspace(0, 4, 12)
b_gv = np.linspace(-1, 3, 12)
WGV, BGV = np.meshgrid(w_gv, b_gv)
DW = np.zeros_like(WGV)
DB = np.zeros_like(BGV)
for i in range(WGV.shape[0]):
    for j in range(WGV.shape[1]):
        dw, db = mse_gradients(WGV[i, j], BGV[i, j], X, y)
        DW[i, j] = dw
        DB[i, j] = db

# Normalise arrows for display (direction only, not magnitude)
mag = np.sqrt(DW**2 + DB**2) + 1e-10
DW_n = DW / mag
DB_n = DB / mag

# Run mini-batch SGD (batch=8) for a nice middle-ground path
history_mini = sgd(X, y, w0, b0, lr=0.05, n_steps=300, batch_size=8)

fig, ax = plt.subplots(figsize=(9, 7))
ax.set_facecolor('#16213e')

# Loss contours (narrower window for clarity)
w_zoom = np.linspace(-0.5, 5, 200)
b_zoom = np.linspace(-2, 4.5, 200)
WZ, BZ = np.meshgrid(w_zoom, b_zoom)
preds_z = WZ[:, :, np.newaxis] * X[np.newaxis, np.newaxis, :] + BZ[:, :, np.newaxis]
LZ = np.mean((y[np.newaxis, np.newaxis, :] - preds_z) ** 2, axis=2)

ax.contourf(WZ, BZ, LZ, levels=40, cmap='inferno', alpha=0.75)
cs = ax.contour(WZ, BZ, LZ, levels=12, colors='white', alpha=0.2, linewidths=0.5)

# Gradient field (negative gradient = descent direction)
ax.quiver(WGV, BGV, -DW_n, -DB_n,
          color='#aaaaaa', alpha=0.35, scale=28, headwidth=3, headlength=4)

# Mini-batch SGD path
ax.plot(history_mini[:, 0], history_mini[:, 1],
        '-', color=BLUE, lw=1, alpha=0.6, label='mini-batch SGD path (batch=8)')

# Full-batch GD path (for comparison)
ax.plot(history_gd[:, 0], history_gd[:, 1],
        '-o', color=GREEN, lw=2, markersize=4, alpha=0.9, label='full-batch GD (clean)')

ax.plot(w0, b0, 's', color=YELLOW, markersize=12, label='start', zorder=6)
ax.plot(TRUE_W, TRUE_B, '*', color='white', markersize=16, label=f'minimum (w={TRUE_W}, b={TRUE_B})', zorder=6)

ax.set_xlabel('w (slope parameter)', fontsize=11)
ax.set_ylabel('b (intercept parameter)', fontsize=11)
ax.set_title('The full picture:\nloss surface + gradient field + GD vs mini-batch SGD paths', color=ACCENT)
ax.legend(fontsize=9, loc='upper right')
ax.set_xlim(-0.5, 5)
ax.set_ylim(-2, 4.5)

plt.tight_layout()
plt.show()

print('Grey arrows: negative gradient direction (descent direction) at each grid point.')
print('The gradient always points perpendicular to the contour lines.')
print('GD follows these arrows exactly (smooth path).')
print('Mini-batch SGD follows noisy estimates of them (jagged path, same destination).')

---

## Summary

**What gradient descent actually is:** an iterative procedure that moves parameters in the direction of steepest loss decrease, by a distance proportional to the learning rate $\eta$.

**What the stochastic part means:** instead of computing the exact gradient using all data (expensive), estimate it using a random sample (cheap). The estimate is unbiased — correct on average — but has variance. Over many steps the noise averages out.

**Why noise can be genuinely useful:** in non-convex landscapes, gradient noise adds enough perturbation to escape shallow local minima and saddle points that clean gradient descent would get stuck in.

**Mini-batch SGD** interpolates between the two: a batch size of $k$ gives $\frac{1}{k}$ the variance of pure SGD, at $k\times$ the per-step computation. Empirically, batch sizes of 32–256 work well for most tasks.

**What we didn't cover:** momentum, adaptive learning rates (Adam, RMSProp), learning rate schedules, second-order methods. These are all responses to specific failure modes of the basic algorithm — oscillation in ill-conditioned landscapes, saddle point trapping, slow convergence in late training.